In [1]:
import pandas as pd

In [2]:
#loading the data
reviews_data = pd.read_csv(r'C:\Users\prith\Downloads\airbnb-data-pipeline\data\raw\reviews.csv.gz')

In [3]:
print(reviews_data.head().to_string())

   listing_id    id        date  reviewer_id reviewer_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [4]:
reviews_data.shape

(1003299, 6)

In [5]:
reviews_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1003299 entries, 0 to 1003298
Data columns (total 6 columns):
 #   Column         Non-Null Count    Dtype
---  ------         --------------    -----
 0   listing_id     1003299 non-null  int64
 1   id             1003299 non-null  int64
 2   date           1003299 non-null  str  
 3   reviewer_id    1003299 non-null  int64
 4   reviewer_name  1003296 non-null  str  
 5   comments       1003028 non-null  str  
dtypes: int64(3), str(3)
memory usage: 45.9 MB


In [6]:
reviews_data.describe()

,listing_id,id,reviewer_id
count,1.003299e+06,1.003299e+06,1.003299e+06
mean,2.508362e+17,6.467927e+17,1.805101e+08
std,4.272822e+17,5.782918e+17,1.765141e+08
min,6.848000e+03,3.149000e+03,1.400000e+01
25%,1.234446e+07,4.640439e+08,3.484390e+07
50%,3.231138e+07,6.877229e+17,1.188337e+08
75%,6.114709e+17,1.166728e+18,2.842070e+08
max,1.635613e+18,1.664626e+18,7.561496e+08


In [7]:
reviews_data.duplicated().sum()

np.int64(0)

In [8]:
Null_percent = pd.DataFrame({
    'missing_percent': reviews_data.isnull().mean() * 100
})

print(Null_percent.to_string())

               missing_percent
listing_id            0.000000
id                    0.000000
date                  0.000000
reviewer_id           0.000000
reviewer_name         0.000299
comments              0.027011


In [9]:
# Summary of the listing dataset

# - The DataFrame has 1003299 rows and 6 columns
# - The data dont have any duplicate value, but it has very few Null values.

# - we have to drop few rows and columns
# - we have to modify the comments to make something useful out of it

In [10]:
#dropping unecessary rows
reviews_data = reviews_data.drop(columns=['id', 'reviewer_id', 'reviewer_name'])

In [11]:
#converting date from str <-> datetime
reviews_data['date'] = pd.to_datetime(reviews_data['date'])

In [12]:
#dropping all null value rows
reviews_data = reviews_data.dropna(axis=0)

In [13]:
print(reviews_data.head().to_string())

   listing_id       date                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [14]:
reviews_data['date'].dtype

dtype('<M8[us]')

In [15]:
reviews_data.shape

(1003028, 3)

In [16]:
Null_percent = pd.DataFrame({
    'missing_percent': reviews_data.isnull().mean() * 100
})

print(Null_percent.to_string())

            missing_percent
listing_id              0.0
date                    0.0
comments                0.0


In [17]:
#modifying comments as a usefull feature
reviews_data['comment_length'] = reviews_data['comments'].str.len()
reviews_agg = reviews_data.groupby('listing_id').agg(
    review_count = ('comments', 'count'),
    avg_comment_length = ('comment_length', 'mean'),
    first_review_date = ('date', 'min'),
    last_review_date = ('date', 'max')
).reset_index()

In [18]:
print(reviews_agg.shape)

(24541, 5)


In [19]:
print(reviews_agg.head(20).to_string())

    listing_id  review_count  avg_comment_length first_review_date last_review_date
0         6848           197          338.055838        2009-05-25       2025-11-17
1         6872             2          279.000000        2022-06-05       2025-10-07
2         6990           251          430.776892        2009-10-28       2026-01-11
3         7097           423          279.810875        2010-01-16       2025-09-23
4         7801            13          479.076923        2009-08-09       2026-03-29
5         8490           189          314.619048        2009-08-25       2023-10-16
6         9357            58          292.396552        2009-10-04       2017-08-13
7        10452            82          381.146341        2010-04-18       2025-03-31
8        12192           316          328.009494        2009-10-27       2026-01-06
9        12940            81          385.728395        2009-12-12       2025-11-15
10       14314           177          397.338983        2009-12-13       202

In [20]:
reviews_agg.to_csv(r'C:\Users\prith\Downloads\airbnb-data-pipeline\data\processed\reviews_clean.csv', index=False)